In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import v2
from transforms3d.axangles import axangle2mat
from transforms3d.euler import quat2euler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import LeaveOneGroupOut
from scipy.interpolate import CubicSpline
from scipy import signal
import more_itertools
import pandas as pd
import numpy as np
import warnings
import cProfile
import pstats
import math
import time
import io
import os
warnings.filterwarnings('ignore')

In [2]:
%load_ext tensorboard
%tensorboard --logdir runs/

Reusing TensorBoard on port 6007 (pid 18604), started 8 days, 21:54:46 ago. (Use '!kill 18604' to kill it.)

In [3]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/duo-gait" # path for kaggle environment
elif os.getenv("COLAB_RELEASE_TAG"):
    from google.colab import drive
    drive.mount('drive')
    data_path = "drive/MyDrive/Colab Notebooks/Datasets" # path for google colab environment
else:
    data_path = "../../data" # path for local environment

In [4]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

**Enable CUDA if available**

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA Version: 12.9


**Custom Transforms**

In [6]:
class ConditionalTransform:
    def __init__(self, transform, enabled=True):
        self.transform = transform
        self.enabled = enabled

    def __call__(self, x):
        if self.enabled:
            return self.transform(x)
        return x

In [7]:
class RandomJitter(object):
    def __init__(self, sigma=0.03):
        self.sigma = sigma

    # https://github.com/terryum/Data-Augmentation-For-Wearable-Sensor-Data/blob/master/Example_DataAugmentation_TimeseriesData.ipynb
    def DA_Jitter(self, X):
        myNoise = np.random.normal(loc=0, scale=self.sigma, size=X.shape)
        return X + myNoise

    def __call__(self, input):
        output = self.DA_Jitter(input.transpose())
        return output.transpose()

In [8]:
class RandomScaling(object):
    def __init__(self, sigma=0.1):
        self.sigma = sigma

    # https://github.com/terryum/Data-Augmentation-For-Wearable-Sensor-Data/blob/master/Example_DataAugmentation_TimeseriesData.ipynb
    def DA_Scaling(self, X, sigma):
        scalingFactor = np.random.normal(loc=1.0, scale=sigma, size=(1, X.shape[1]))
        myNoise = np.matmul(np.ones((X.shape[0], 1)), scalingFactor)
        return X * myNoise

    def __call__(self, input):
        output = self.DA_Scaling(input.transpose(), self.sigma)
        return output.transpose()

In [9]:
class RandomRotation(object):
    def __init__(self, degree=10):
        self.rad = np.deg2rad(degree)

    # https://github.com/terryum/Data-Augmentation-For-Wearable-Sensor-Data/blob/master/Example_DataAugmentation_TimeseriesData.ipynb
    def DA_Rotation(self, X):
        angle = np.random.uniform(low=-self.rad, high=self.rad)
        axis = np.random.uniform(low=-1, high=1, size=3)
        return np.matmul(X, axangle2mat(axis, angle))

    def __call__(self, input): # TODO - consider applying rotation per sensor instead of every 3 channels
        input = input.transpose()
        output = np.zeros_like(input)
        channels = input.shape[1]
        for i in range(channels//3):
            output[:,i*3:i*3+3] = self.DA_Rotation(input[:,i*3:i*3+3])
        return output.transpose()

In [10]:
class RandomMagWarp(object):
    def __init__(self, sigma=0.1, knot=4):
        self.sigma = sigma
        self.knot = knot

    # https://github.com/uchidalab/time_series_augmentation/blob/master/utils/augmentation.py
    def magnitude_warp(self, x):
        orig_steps = np.arange(x.shape[1])
        
        random_warps = np.random.normal(loc=1.0, scale=self.sigma, size=(x.shape[0], self.knot+2, x.shape[2]))
        warp_steps = (np.ones((x.shape[2],1))*(np.linspace(0, x.shape[1]-1., num=self.knot+2))).T
        ret = np.zeros_like(x)
        for i, pat in enumerate(x):
            warper = np.array([CubicSpline(warp_steps[:,dim], random_warps[i,:,dim])(orig_steps) for dim in range(x.shape[2])]).T
            ret[i] = pat * warper

        return ret

    def __call__(self, input):
        input = input.transpose()
        input = np.expand_dims(input, axis=0)

        output = self.magnitude_warp(input)
        output = np.squeeze(output, axis=0)
        return output.transpose()

In [11]:
class RandomTimeWarp(object):
    def __init__(self, sigma=0.1, knot=4):
        self.sigma = sigma
        self.knot = knot
    
    # https://github.com/uchidalab/time_series_augmentation/blob/master/utils/augmentation.py
    def time_warp(self, x):
        orig_steps = np.arange(x.shape[1])
        
        random_warps = np.random.normal(loc=1.0, scale=self.sigma, size=(x.shape[0], self.knot+2, x.shape[2]))
        warp_steps = (np.ones((x.shape[2],1))*(np.linspace(0, x.shape[1]-1., num=self.knot+2))).T
        
        ret = np.zeros_like(x)
        for i, pat in enumerate(x):
            for dim in range(x.shape[2]):
                time_warp = CubicSpline(warp_steps[:,dim], warp_steps[:,dim] * random_warps[i,:,dim])(orig_steps)
                scale = (x.shape[1]-1)/time_warp[-1]
                ret[i,:,dim] = np.interp(orig_steps, np.clip(scale*time_warp, 0, x.shape[1]-1), pat[:,dim]).T
        return ret

    def __call__(self, input):
        input = input.transpose()
        input = np.expand_dims(input, axis=0)

        output = self.time_warp(input)
        output = np.squeeze(output, axis=0)
        return output.transpose()

In [12]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input):
        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)
        return smoothed_imu_signal

In [13]:
class PadTrimToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        input = torch.from_numpy(input.copy()) # convert to tensor

        C, T = input.shape

        if T == self.padlen:
            return input

        return F.pad(input, (0, self.padlen - T), value=0)

In [14]:
class Normalize(object):
    def __init__(self, mean, std, epsilon=1e-7):
        self.mean = mean
        self.std = std
        self.epsilon = epsilon

    def __call__(self, input):
        output = (input - self.mean) / (self.std + self.epsilon)
        return output

**Define dataset pipeline**

In [15]:
class DUO_GAIT(Dataset):
    def __init__(self, freq, window_size, allowed_sensors, num_participants, ignore_participant_ids, compute_jerk=True, include_angle=True, include_dt=True, remove_outliers=True, transform=None, target_transform=None):
        self.num_participants = num_participants
        self.allowed_sensors = allowed_sensors
        self.foot_strides_df = None
        self.imu_signals = []
        self.freq = freq
        self.window_samples_size = window_size
        
        self.include_angle = include_angle
        self.include_dt = include_dt

        sensor_dict = {}
        for participant_id in range(1, num_participants+1):
            if participant_id in ignore_participant_ids:
                continue

            for sensor_location in self.allowed_sensors:
                for protocol in ["control", "fatigue"]:
                    for task in ["st", "dt"]:
                        if task == "dt" and not include_dt: # skip dual task if we don't want it
                            continue

                        interim_file_name = f"{data_path}/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{participant_id:02}/{sensor_location}.csv"
                        interim_sensor_df = pd.read_csv(interim_file_name)
                        interim_sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)

                        if include_angle:
                            raw_file_name = f"{data_path}/DUO-GAIT/raw/OG_{task}_raw/sub_{participant_id:02}/{sensor_location}.csv"
                            raw_sensor_df = pd.read_csv(raw_file_name,skiprows=5).drop(index=0).astype(float)
                            raw_sensor_df = raw_sensor_df.filter(["Quat W", "Quat X", "Quat Y", "Quat Z"], axis=1)
                            raw_sensor_df.rename({ "Quat W": "QuatW", "Quat X": "QuatX", "Quat Y": "QuatY", "Quat Z": "QuatZ" },axis=1, inplace=True, errors='raise')

                            quat_features_df = raw_sensor_df.loc[interim_sensor_df['Sample'].min(): interim_sensor_df['Sample'].max()].reset_index(drop=True)
                            euler_angles = quat_features_df.apply(lambda row: quat2euler(row, axes='sxyz'), axis=1)
                            euler_features_df = pd.DataFrame(euler_angles.tolist(), index=euler_angles.index, columns=['EulerX', 'EulerY', 'EulerZ']).reset_index(drop=True) # get euler angles
                            
                            interim_sensor_df = pd.merge(interim_sensor_df, euler_features_df, left_index=True, right_index=True, how='inner')

                        sensor_dict[interim_file_name] = interim_sensor_df
            # save interim files for future

            for protocol in ["control", "fatigue"]:
                for task in ["st", "dt"]:
                    if task == "dt" and not include_dt: # skip dual task if we don't want it
                        continue

                    min_sample_number, max_sample_number = 1e9, 0

                    for foot in ["left", "right"]:
                        foot_df = pd.read_csv(f"{data_path}/DUO-GAIT/processed/OG_{task}_{protocol}/sub_{participant_id:02}/{foot}_foot_core_params.csv")
                        foot_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
                        foot_df['is_fatigue'] = int(protocol == "fatigue")
                        foot_df['is_dual_task'] = int(task == "dt")
                        foot_df['Participant'] = participant_id

                        is_control = (protocol == "control")
                        is_dual_task = (task == "dt")
                        
                        self.create_start_end_samples_strides(sensor_dict, foot_df, participant_id, is_control=is_control, is_dual_task=is_dual_task)

                        min_sample_number = min(min_sample_number, foot_df['start_samples'].min())
                        max_sample_number = max(max_sample_number, foot_df['end_samples'].max())

                    is_fatigue = int(protocol == "fatigue")
                    is_dual_task = int(task == "dt")

                    sample_ranges = np.arange(min_sample_number, max_sample_number+1)
                    windows = more_itertools.windowed(sample_ranges, n=self.window_samples_size, step=self.window_samples_size//2)
                    
                    for window in windows:
                        window = [item for item in window if item is not None] # remove all none type
                        window_df = pd.DataFrame([{'is_fatigue': is_fatigue, 'is_dual_task': is_dual_task, 'Participant': participant_id, 'start_samples': window[0], 'end_samples': window[-1]}])
                        self.foot_strides_df = pd.concat([self.foot_strides_df, window_df], axis=0, ignore_index=True)

        for _, row in self.foot_strides_df.iterrows():
            imu_signals_df_list = []
            for sensor_location in self.allowed_sensors:
                protocol = "fatigue" if row['is_fatigue'] else "control"
                task = "dt" if row['is_dual_task'] else "st"
                file_name = f"{data_path}/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{row['Participant']:02}/{sensor_location}.csv"

                sensor_df = sensor_dict[file_name]
                sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])]

                acc_columns = ["AccX", "AccY", "AccZ"]
                gyr_columns = ["GyrX", "GyrY", "GyrZ"]
                angle_columns = ["EulerX", "EulerY", "EulerZ"] if include_angle else []

                sensor_signals_df = sensor_df[acc_columns + gyr_columns + angle_columns].reset_index(drop=True)
                sensor_signals_df = sensor_signals_df.add_prefix(f"{sensor_location}_")

                if compute_jerk:
                    acc_gyro_df = sensor_df[acc_columns + gyr_columns]
                    acc_gyro_np = acc_gyro_df.to_numpy().transpose().astype(np.float32)
                    jerk_np = np.gradient(acc_gyro_np, 1.0 / self.freq, axis=1).transpose()
                    jerk_df = pd.DataFrame(data=jerk_np, columns=["Jerk" + col for col in (acc_columns + gyr_columns)])
                    jerk_df = jerk_df.add_prefix(f"{sensor_location}_")
                    sensor_signals_df = pd.concat([sensor_signals_df, jerk_df], axis=1)

                imu_signals_df_list.append(sensor_signals_df)

            imu_signals_df = pd.concat(imu_signals_df_list, axis=1)
            imu_signals_np = imu_signals_df.to_numpy().transpose().astype(np.float32)
            self.imu_signals.append(imu_signals_np)

        self.num_channels = self.imu_signals[0].shape[0]
        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, sensor_dict, df, participant_id, is_control, is_dual_task):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"
        task = "dt" if is_dual_task else "st"

        file_name = f"{data_path}/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{participant_id:02}/{self.allowed_sensors[0]}.csv"        
        fatigue_df = sensor_dict[file_name].copy() # copy to avoid changing sensor dict's dataframes
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_np = self.imu_signals[idx]
        label = row["is_fatigue"]

        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

**DeepConvLSTM Architecture**

In [16]:
class DeepConvLSTM(nn.Module):
    def __init__(self, channels, p_drop):
        super().__init__()
        self.features = nn.Sequential(
                        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(10, 3), stride=(1,1), padding='same'),
                        nn.ReLU(),
                        nn.Dropout(p_drop),

                        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(10, 3), stride=(1,1), padding='same'),
                        nn.ReLU(),
                        nn.Dropout(p_drop))
        
        self.lstm1 = nn.LSTM(input_size=channels*64, hidden_size=64, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)
        self.drop = nn.Dropout(p_drop)
        
        self.dense = nn.Sequential(nn.Linear(in_features=128, out_features=128),
                                   nn.ReLU(),
                                   nn.Dropout(p_drop),
                                   nn.Linear(in_features=128, out_features=2)
                                   )

    def forward(self, x):
        B, C, T = x.shape # (B, C, T)

        x = x.permute(0, 2, 1) # (B, T, C)
        x = x.unsqueeze(1) # (B, 1, T, C)
        x = self.features(x) # (B, 64, T, C)
        
        x = x.permute(0, 2, 3, 1) # (B, T, C, 64)
        x = x.reshape(B, T, C * 64) # (B, T, C * 64)

        x, _ = self.lstm1(x) # (B, T, 64)
        x = self.drop(x)
        x, _ = self.lstm2(x) # (B, T, 128)
        x = self.drop(x)

        x = x[:, -1, :] # (B, 128)
        x = self.dense(x) # (B, 2)

        return x

**Experiment Parameters**

In [ ]:
num_participants = 18
ignore_participant_ids = [4, 7, 16]
allowed_sensors = ["LL", "RL"]
window_size = 128
num_samples = window_size
freq = 128

**Hyper Parameters**

In [ ]:
batch_size = 32
epochs = 50
learning_rate = 0.0003
min_learning_rate = 0.0001
patience_epochs = 20
learning_rate_factor = 0.9
early_stopping_rounds = 25
train_percent = 0.90
p_drop = 0.50
label_smooth = 0.1
weight_decay = 0

**Transform Parameters**

In [19]:
enable_butterworth_filter = True
enable_random_jitter = True
enable_random_scaling = False
enable_random_rotation = False
enable_random_timewarp = False
enable_random_magwarp = False

transform_dict = {
    'filter_butterw': enable_butterworth_filter,
    'random_jitter': enable_random_jitter,
    'random_scale': enable_random_scaling,
    'random_rot': enable_random_rotation,
    'random_timewarp': enable_random_timewarp,
    'random_magwarp': enable_random_magwarp,
}

enabled_transforms = []
for key, is_enabled in transform_dict.items():
    if is_enabled:
        enabled_transforms.append(key)

**Leave one Participant out Training Loop**

In [20]:
print("Loading Dataset ...")
dataset = DUO_GAIT(freq, window_size, allowed_sensors, num_participants, ignore_participant_ids, include_angle=False, include_dt=False)
print("Dataset Loaded ...")

dataset_size = len(dataset)
groups = dataset.foot_strides_df['Participant'].to_numpy()

gss = GroupShuffleSplit(n_splits=1, train_size=train_percent, random_state=42)
train_val_idx, test_idx = next(gss.split(X=dataset, groups=groups))

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

groups_train_val = groups[train_val_idx]

train_lopo_accuracies = []
valid_lopo_accuracies = []

logo = LeaveOneGroupOut()
for i, (train_index, valid_index) in enumerate(logo.split(X=train_val_set, groups=groups_train_val)):
    train_set = Subset(train_val_set, train_index)
    valid_set = Subset(train_val_set, valid_index)

    dataset.transform = v2.Compose([
        ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
        PadTrimToLength(padlen=num_samples),
        v2.Lambda(lambda x: x.to(torch.float32))
    ]) # default transform before normalization

    stat_loader = DataLoader(train_set, batch_size=batch_size)
    lopo = groups_train_val[valid_index][0]

    full_train_data = []
    for batch_idx, (train_features, train_labels) in enumerate(stat_loader):
        full_train_data.append(train_features)

    full_train_data = torch.concat(full_train_data, dim=0)
    std, mean = torch.std_mean(full_train_data, dim=(0, 2), keepdim=True)
    # compute mean and std over all channels separately

    std = torch.squeeze(std, dim=0)
    mean = torch.squeeze(mean, dim=0)
    # remove the batch dimension

    model = DeepConvLSTM(dataset.num_channels, p_drop).to(device=device) # move model to device

    timestamp = time.strftime("%b-%d-%Y %I-%M-%S %p")
    
    writer = SummaryWriter(log_dir=f"runs/{timestamp}/lopo_{lopo:02} p_drop_{p_drop} lr_{learning_rate} wd_{weight_decay} label_smooth_{label_smooth} {' '.join(allowed_sensors)} DeepConvLSTM compute_jerk {window_size//128}s_windows {'st_dt' if dataset.include_dt else 'st'} {'joint_angles' if dataset.include_angle else ''} {' '.join(enabled_transforms)}", flush_secs=30)

    loss = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=learning_rate_factor, patience=patience_epochs, min_lr=min_learning_rate)

    epochs_without_gain = 0
    best_valid_loss = float("inf")

    max_train_acc = 0
    max_valid_acc = 0

    print(f"Beginning training for lopo_{lopo:02}")
    for epoch in range(0, epochs):
        train_epoch_loss, valid_epoch_loss = 0.0, 0.0

        train_epoch_preds, train_epoch_labels = [], []
        valid_epoch_preds, valid_epoch_labels = [], []

        dataset.transform = v2.Compose([
            ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
            ConditionalTransform(RandomJitter(sigma=0.03),enabled=enable_random_jitter),
            ConditionalTransform(RandomScaling(sigma=0.2),enabled=enable_random_scaling),
            ConditionalTransform(RandomRotation(degree=10),enabled=enable_random_rotation),
            ConditionalTransform(RandomTimeWarp(sigma=0.2, knot=4),enabled=enable_random_timewarp),
            ConditionalTransform(RandomMagWarp(sigma=0.2, knot=4),enabled=enable_random_magwarp),
            PadTrimToLength(padlen=num_samples),
            v2.Lambda(lambda x: x.to(torch.float32)),
            Normalize(mean=mean, std=std),
        ]) # update transform for normalization

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
        # create dataloader for training

        train_start_time = time.time()
        model.train()
        for batch_idx, (train_features, train_labels) in enumerate(train_loader):
            train_batch_size = len(train_labels)
            train_features = train_features.to(device)
            train_labels = train_labels.to(device) # move data to device

            optimizer.zero_grad()

            logits = model(train_features)
            predictions = torch.argmax(logits, dim=1)
            
            train_epoch_preds.extend(predictions.cpu().numpy())
            train_epoch_labels.extend(train_labels.cpu().numpy())

            train_batch_loss = loss(logits, train_labels)
            train_batch_loss.backward()

            optimizer.step()

            train_epoch_loss += train_batch_loss.item() * train_batch_size
        train_end_time = time.time()

        dataset.transform = v2.Compose([
            ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
            PadTrimToLength(padlen=num_samples),
            v2.Lambda(lambda x: x.to(torch.float32)),
            Normalize(mean=mean, std=std),
        ]) # update transform for normalization

        valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, pin_memory=True)
        # create dataloader for validation

        model.eval()
        valid_start_time = time.time()
        with torch.no_grad():
            for batch_idx, (valid_features, valid_labels) in enumerate(valid_loader):
                valid_batch_size = len(valid_labels)
                
                valid_features = valid_features.to(device)
                valid_labels = valid_labels.to(device) # move to device
                
                logits = model(valid_features)
                predictions = torch.argmax(logits, dim=1)

                valid_epoch_preds.extend(predictions.cpu().numpy())
                valid_epoch_labels.extend(valid_labels.cpu().numpy())

                valid_batch_loss = loss(logits, valid_labels)
                valid_epoch_loss += valid_batch_loss.item() * valid_batch_size
        valid_end_time = time.time()

        train_epoch_loss /= len(train_set)
        valid_epoch_loss /= len(valid_set)

        scheduler.step(valid_epoch_loss) # step the scheduler and reduce lr if necessary

        train_epoch_acc = accuracy_score(train_epoch_labels, train_epoch_preds)
        valid_epoch_acc = accuracy_score(valid_epoch_labels, valid_epoch_preds)

        max_train_acc = max(max_train_acc, train_epoch_acc)
        max_valid_acc = max(max_valid_acc, valid_epoch_acc)

        writer.add_scalar("Learning Rate/epoch", scheduler.get_last_lr()[0], epoch)

        writer.add_scalar("Loss/train-epoch", train_epoch_loss, epoch)
        writer.add_scalar('Accuracy/train-epoch', train_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/train-epoch-secs', train_end_time - train_start_time, epoch)

        writer.add_scalar("Loss/valid-epoch", valid_epoch_loss, epoch)
        writer.add_scalar('Accuracy/valid-epoch', valid_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/valid-epoch-secs', valid_end_time - valid_start_time, epoch)

        if valid_epoch_loss < best_valid_loss:
            best_valid_loss = valid_epoch_loss
            epochs_without_gain = 0
        else:
            epochs_without_gain += 1

        if epochs_without_gain >= early_stopping_rounds:
            break

    print(f"Train Acc for lopo_{lopo:02}:", max_train_acc)
    print(f"Valid Acc for lopo_{lopo:02}:", max_valid_acc)

    train_lopo_accuracies.append(max_train_acc)
    valid_lopo_accuracies.append(max_valid_acc)

train_lopo_accuracies = np.array(train_lopo_accuracies)
valid_lopo_accuracies = np.array(valid_lopo_accuracies)

print("Train Acc:", np.mean(train_lopo_accuracies), "+=", np.std(train_lopo_accuracies))
print("Valid Acc:", np.mean(valid_lopo_accuracies), "+=", np.std(valid_lopo_accuracies))

Loading Dataset ...
Dataset Loaded ...
Beginning training for lopo_02


KeyboardInterrupt: 

In [ ]:
dataset.transform = v2.Compose([
    ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
    PadTrimToLength(padlen=num_samples),
    v2.Lambda(lambda x: x.to(torch.float32))
]) # default transform before normalization

stat_loader = DataLoader(train_val_set, batch_size=batch_size)

full_train_val_data = []
for batch_idx, (train_features, train_labels) in enumerate(stat_loader):
    full_train_val_data.append(train_features)

full_train_val_data = torch.concat(full_train_val_data, dim=0)
std, mean = torch.std_mean(full_train_val_data, dim=(0, 2), keepdim=True)
# compute mean and std over all channels separately

std = torch.squeeze(std, dim=0)
mean = torch.squeeze(mean, dim=0)
# remove the batch dimension

dataset.transform = v2.Compose([
    ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
    ConditionalTransform(RandomJitter(sigma=0.03),enabled=enable_random_jitter),
    ConditionalTransform(RandomScaling(sigma=0.2),enabled=enable_random_scaling),
    ConditionalTransform(RandomRotation(degree=10),enabled=enable_random_rotation),
    ConditionalTransform(RandomTimeWarp(sigma=0.2, knot=4),enabled=enable_random_timewarp),
    ConditionalTransform(RandomMagWarp(sigma=0.2, knot=4),enabled=enable_random_magwarp),
    PadTrimToLength(padlen=num_samples),
    v2.Lambda(lambda x: x.to(torch.float32)),
    Normalize(mean=mean, std=std),
]) # update transform for normalization

train_val_loader = DataLoader(train_val_set, batch_size=batch_size, shuffle=True, pin_memory=True)
# create dataloader for training + validation

model = DeepConvLSTM(dataset.num_channels, p_drop).to(device=device) # move model to device

loss = nn.CrossEntropyLoss(label_smoothing=label_smooth)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
epochs = 15 # much lower epochs to avoid overfitting

print(f"Beginning training for testing setup")
for epoch in range(0, epochs):
    model.train()
    for batch_idx, (train_features, train_labels) in enumerate(train_val_loader):
        train_batch_size = len(train_labels)

        train_features = train_features.to(device)
        train_labels = train_labels.to(device) # move data to device

        optimizer.zero_grad()

        logits = model(train_features)
        predictions = torch.argmax(logits, dim=1)
        
        train_batch_loss = loss(logits, train_labels)
        train_batch_loss.backward()

        optimizer.step()

dataset.transform = v2.Compose([
    ConditionalTransform(ButterworthFilter(cutoff=10.0, order=3, fs=128.0), enabled=enable_butterworth_filter),
    PadTrimToLength(padlen=num_samples),
    v2.Lambda(lambda x: x.to(torch.float32)),
    Normalize(mean=mean, std=std),
]) # update transform for normalization

test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=True)
# create dataloader for testing

test_full_preds, test_full_labels = [], []

model.eval()
with torch.no_grad():
    for batch_idx, (test_features, test_labels) in enumerate(test_loader):
        test_batch_size = len(test_labels)
        
        test_features = test_features.to(device)
        test_labels = test_labels.to(device) # move to device
        
        logits = model(test_features)
        predictions = torch.argmax(logits, dim=1)

        test_full_preds.extend(predictions.cpu().numpy())
        test_full_labels.extend(test_labels.cpu().numpy())
test_acc = accuracy_score(test_full_labels, test_full_preds)
print("Test Acc:", test_acc)

Beginning training for testing setup
Test Acc: 0.7174917491749175
